# Debug List Scraper Berita Malang
List-only. URL debug: `https://beritamalang.media/category/berita/`. Tidak scrape artikel detail.


In [1]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, parse_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.beritamalang as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/beritamalang.py


In [2]:
MAX_PAGES = 200
TITLE_LIMIT = 90



def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = pd.to_datetime(
        df['published_date'].apply(parse_date),
        errors='coerce',
    )
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    has_page = df['page_num'].notna().any()
    has_date = df['published_dt'].notna().any()

    print('first page:', df['page_num'].dropna().min() if has_page else None)
    print('last page:', df['page_num'].dropna().max() if has_page else None)
    print('newest date:', df['published_dt'].max() if has_date else None)
    print('oldest date:', df['published_dt'].min() if has_date else None)
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()) if has_date else False)
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    if has_date:
        display(
            df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
            .groupby('month', dropna=False)
            .size()
            .reset_index(name='count')
        )
    else:
        print('No parseable published_date in list page.')

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [3]:
BERITAMALANG_BASE_URL = 'https://beritamalang.media/category/berita/'


def berita_page_url(page_num):
    if page_num == 1:
        return BERITAMALANG_BASE_URL
    return f'{BERITAMALANG_BASE_URL}page/{page_num}/'


def debug_scrape_list(cutoff, max_pages=MAX_PAGES):
    rows = []
    for page_num in range(1, max_pages + 1):
        url = berita_page_url(page_num)
        print(f'Scraping Berita Malang page {page_num}: {url}')
        try:
            soup = fetch_html(url)
        except Exception as error:
            print(f'Gagal buka Berita Malang page {page_num}: {error}')
            break

        cards = soup.select('article.post-main')
        if not cards:
            break

        stop = False
        for card in cards:
            title_tag = card.select_one('h2.post-main-title a')
            date_tag = card.select_one('.post-main-datapost span')
            category_tag = card.select_one('.post-main-category')
            if not title_tag:
                continue

            published_date = date_tag.get_text(strip=True) if date_tag else None
            if is_older_than_cutoff(published_date, cutoff):
                stop = True
                break

            rows.append({
                'page_num': page_num,
                'list_page_url': url,
                'title': title_tag.get_text(strip=True),
                'url': title_tag['href'],
                'published_date': published_date,
                'category_raw': category_tag.get_text(',', strip=True) if category_tag else None,
            })
        if stop:
            break
    return records_to_df(rows)


list_df = debug_scrape_list(cutoff)
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)


Scraping Berita Malang page 1: https://beritamalang.media/category/berita/
Scraping Berita Malang page 2: https://beritamalang.media/category/berita/page/2/
Scraping Berita Malang page 3: https://beritamalang.media/category/berita/page/3/
Scraping Berita Malang page 4: https://beritamalang.media/category/berita/page/4/
Scraping Berita Malang page 5: https://beritamalang.media/category/berita/page/5/
Scraping Berita Malang page 6: https://beritamalang.media/category/berita/page/6/
Scraping Berita Malang page 7: https://beritamalang.media/category/berita/page/7/
Scraping Berita Malang page 8: https://beritamalang.media/category/berita/page/8/
page=001 | date=June 28, 2026 | title=Kapolres Batu Gelar Turnamen Billiard Kapolres Cup Sambut HUT Bhayangkara ke-80
page=001 | date=June 27, 2026 | title=Sepiring Berkah dari Ladang Sendiri: Hangatnya Atur Pisungsung di Desa Tulungrejo
page=001 | date=June 26, 2026 | title=Festival Jenang Suro Bumiaji Perkuat Gotong Royong dan Ekonomi Lokal
page=0

In [4]:
list_df = audit_list(list_df)


total rows: 70
first page: 1
last page: 7
newest date: 2026-06-28 00:00:00
oldest date: 2026-03-02 00:00:00
cutoff date: 2026-02-28 12:22:47.265456
reached cutoff: False
null parsed dates: 0

rows per month:


,month,count
0,2026-03,21
1,2026-04,24
2,2026-05,9
3,2026-06,16



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-28,2026-06-12
1,2,10,2026-06-12,2026-05-27
2,3,10,2026-05-27,2026-04-23
3,4,10,2026-04-23,2026-04-14
4,5,10,2026-04-13,2026-03-31
5,6,10,2026-03-28,2026-03-20
6,7,10,2026-03-19,2026-03-02


In [5]:
output_path = PROJECT_ROOT / 'csv' / 'beritamalang_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/beritamalang_list_debug.csv')